# Notebook 3: PV-Potenzial – Adress- und Gemeindenvergleich Schweiz

Dieses Notebook kombiniert die PVOUT-Solardaten (aus Notebook 1) mit den Stromtarifen (aus Notebook 2)
und berechnet den **PV-Attraktivitätsscore** für jede Schweizer Gemeinde.

**Formel:** `Score = PVOUT [kWh/kWp/Jahr] × Stromtarif [Rp./kWh]`  
**Ergebnis:** Rp/kWp/Jahr – wie viel Strom pro installierter Leistungseinheit jährlich eingespart wird.

Beispiel: Score 14 000 Rp/kWp/Jahr → 5-kWp-Anlage spart ca. **700 CHF/Jahr**.

**Warum Score = PVOUT × Tarif?**
- PVOUT = kWh/kWp/Jahr (wie viel Strom eine Anlage produziert)
- Stromtarif = Rp./kWh (wie viel der Strom kostet)
- Score = Rp/kWp/Jahr (wie viel man durch Eigenverbrauch spart)

Wichtig: Eine sonnige Gemeinde mit **günstigem Strom** kann einen tieferen Score haben als eine
weniger sonnige Gemeinde mit **teurem Strom** – das ist korrekt! PV lohnt sich dort mehr, wo Strom teuer ist.

**Input-Daten:**
- `pvout_gemeinden.csv` – mittlerer PVOUT pro Gemeinde (aus Notebook 1)
- `df_h4_processed.csv` – Stromtarife H4-Haushalt pro Gemeinde (aus Notebook 2)
- `PVOUT.tif` – Solardaten-Raster für adressgenaue Pixelabfrage
- `swissBOUNDARIES3D.gpkg` – Gemeindegrenzen für Karte und Spatial Join

---

## Schritt 1: Google Drive verbinden und Bibliotheken installieren

**Ziel:** Zugriff auf die Dateien im Google Drive herstellen und alle benötigten Python-Pakete laden.

**Was der Code macht:**
- `drive.mount` hängt das Google Drive unter `/content/drive/MyDrive/` ein – danach kann Python direkt auf alle Dateien zugreifen
- `pip install` lädt fehlende Pakete nach

**Bibliotheken und ihre Aufgaben in diesem Notebook:**

| Bibliothek | Wird verwendet in | Aufgabe |
|---|---|---|
| `pandas` | Schritt 3, 4, 11 | Tabellen laden, Daten zusammenführen (Join), Ranking erstellen |
| `rasterio` | Schritt 8 | PVOUT.tif öffnen und Pixelwert an bestimmten Koordinaten ablesen |
| `geopandas` | Schritt 5, 9, 12 | Gemeindegrenzen laden, Koordinatensystem umrechnen, Karten zeichnen |
| `shapely` | Schritt 9 | Punkt-Objekt aus Koordinaten erstellen (für Spatial Join benötigt) |
| `folium` | Schritt 12 | Interaktive Karte mit Choropleth und Adress-Marker erstellen |
| `fiona` | Schritt 5 | Treiber zum Öffnen von `.gpkg`-Dateien (GeoPackage) |
| `requests` | Schritt 7 | HTTP-Anfrage an die swisstopo Geocoding-API |
| `geopy` | Schritt 7 | Geocoding-Fallback via Nominatim/OpenStreetMap |
| `glob`, `os`, `shutil`, `zipfile` | Schritt 2 | Dateien im Drive suchen, kopieren und entpacken |

In [72]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [73]:
!pip install rasterio geopandas folium fiona geopy --quiet

In [74]:
# Definieren der Pfade zu den Quelldateien
# URLs der Rohdaten im GitHub-Repository anzeigen
PVOUT_PATH = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/PVOUT.tif'
GPKG_PATH  = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/swissBOUNDARIES3D_1_5_LV95_LN02.gpkg'
PVOUT_CSV_PATH = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/pvout_gemeinden.csv'
TARIFE_PATH = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/df_h4_processed.csv'

print(f'Defined PVOUT.tif path: {PVOUT_PATH}')
print(f'Defined GPKG path:      {GPKG_PATH}')
print(f'Defined pvout_gemeinden.csv path: {PVOUT_CSV_PATH}')
print(f'Defined df_h4_processed.csv path: {TARIFE_PATH}')

Defined PVOUT.tif path: https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/PVOUT.tif
Defined GPKG path:      https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/swissBOUNDARIES3D_1_5_LV95_LN02.gpkg
Defined pvout_gemeinden.csv path: https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/pvout_gemeinden.csv
Defined df_h4_processed.csv path: https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/df_h4_processed.csv


## Schritt 2: Dateien automatisch suchen

**Ziel:** Alle vier benötigten Dateien finden, egal wo sie im Google Drive gespeichert sind.

**Was sind diese Dateien?**

**pvout_gemeinden.csv** – Output aus Notebook 1  
Enthält den mittleren PVOUT-Wert (kWh/kWp/Jahr) für jede der 2132 Schweizer Gemeinden, berechnet aus dem Global Solar Atlas.

**df_h4_processed.csv** – Output aus Notebook 2  
Enthält den H4-Haushaltsstromtarif (Rp./kWh, inkl. Netznutzung, Energie, Abgaben) für ca. 2100 Gemeinden aus den ElCom-Daten.
Bei Gemeinden mit mehreren Netzbetreibern wird der höchste Tarif verwendet.

**PVOUT.tif** – Solardaten-Raster (Global Solar Atlas v2)  
Wird für die adressgenaue Pixelabfrage in Schritt 8 benötigt. Auflösung: ca. 1 km × 1 km.

**swissBOUNDARIES3D.gpkg** – Gemeindegrenzen (swisstopo)  
Polygone aller Schweizer Gemeinden. Werden für den Spatial Join (Schritt 9) und die Choropleth-Karte (Schritt 12) benötigt.

**Was der Code macht:**
- `suche_datei(muster)` durchsucht alle Unterordner von Google Drive nach einer Datei mit diesem Namen
- `suche_gpkg()` hat drei Stufen: direkte .gpkg-Datei → Ordner mit .gpkg-Endung → ZIP-Archiv entpacken
- Am Ende stehen vier Pfad-Variablen für alle weiteren Schritte bereit

In [75]:
import pandas as pd

url = 'https://raw.githubusercontent.com/nesasad/BINA_PV_Anlagen/main/Data/Rohdaten-Tarife-Standard-Produkt.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(url)

# Display the first few rows of the DataFrame to confirm
df.head()

,uid,netzbetreiber,kategorieName,gemeindeNummer,gemeindeName,kanton,total,energie,abgaben,netznutzung,netzzuschlag,messwesen
0,CHE-105.981.944,AEW Energie AG,C1,4172,Münchwilen (AG),AG,26.594,11.4,0.689,11.53,2.3,54.0
1,CHE-105.981.944,AEW Energie AG,C1,4253,Magden,AG,26.594,11.4,0.689,11.53,2.3,54.0
2,CHE-105.981.944,AEW Energie AG,C1,4106,Mönthal,AG,26.555,11.4,0.650,11.53,2.3,54.0
3,CHE-105.981.944,AEW Energie AG,C1,4068,Hägglingen,AG,26.594,11.4,0.689,11.53,2.3,54.0
4,CHE-105.981.944,AEW Energie AG,C1,4258,Rheinfelden,AG,26.594,11.4,0.689,11.53,2.3,54.0


## Schritt 3: Daten laden und Überblick verschaffen

**Ziel:** Beide CSV-Dateien in pandas DataFrames laden und die Spaltenstruktur verstehen.

**Was sind die Spalten?**

**pvout_gemeinden.csv:**

| Spalte | Typ | Beschreibung |
|---|---|---|
| `gemeinde_nr` | int | BFS-Gemeindenummer (eindeutiger Schlüssel für den Join) |
| `gemeinde_name` | str | Offizieller Gemeindename |
| `kanton_name` | str | Deutscher Kantonsname |
| `pvout_kwh_kwp_year` | float | Mittlerer PVOUT-Wert der Gemeinde [kWh/kWp/Jahr] |

**df_h4_processed.csv:**

| Spalte | Typ | Beschreibung |
|---|---|---|
| `netzbetreiber` | str | Name des Stromnetzbetreibers |
| `gemeindeNummer` | int | BFS-Gemeindenummer (Join-Schlüssel) |
| `gemeindeName` | str | Gemeindename |
| `kanton` | str | Kantonskürzel (z.B. BE, ZH) |
| `total` | float | Gesamtstromtarif H4 [Rp./kWh] – inkl. Energie, Netz, Abgaben |

**Join-Schlüssel:** `pvout_gemeinden.gemeinde_nr` ↔ `df_h4_processed.gemeindeNummer` (beide int, BFS-Nummer)

In [76]:
import pandas as pd

pvout  = pd.read_csv(PVOUT_CSV_PATH)
tarife = pd.read_csv(TARIFE_PATH)

print('=== pvout_gemeinden.csv ===')
print(f'Zeilen: {len(pvout)}, Spalten: {list(pvout.columns)}')
print(pvout.head(3).to_string(index=False))

print()
print('=== df_h4_processed.csv ===')
print(f'Zeilen: {len(tarife)}, Spalten: {list(tarife.columns)}')
print(tarife.head(3).to_string(index=False))

=== pvout_gemeinden.csv ===
Zeilen: 2132, Spalten: ['gemeinde_nr', 'gemeinde_name', 'kanton_name', 'pvout_kwh_kwp_year']
 gemeinde_nr gemeinde_name kanton_name  pvout_kwh_kwp_year
         131      Adliswil      Zürich         1155.791391
        3714     Rheinwald  Graubünden         1273.649336
        5722         Grens       Waadt         1292.010905

=== df_h4_processed.csv ===
Zeilen: 2123, Spalten: ['netzbetreiber', 'gemeindeNummer', 'gemeindeName', 'kanton', 'total']
                              netzbetreiber  gemeindeNummer       gemeindeName kanton  total
Elektrizitätswerke des Kantons Zürich (EKZ)               1    Aeugst am Albis     ZH 24.136
Elektrizitätswerke des Kantons Zürich (EKZ)               2 Affoltern am Albis     ZH 24.136
Elektrizitätswerke des Kantons Zürich (EKZ)               3         Bonstetten     ZH 24.136


## Schritt 4: Datenqualität prüfen (Qualitätskontrolle 1)

**Ziel:** Sicherstellen, dass die Daten vollständig und plausibel sind, bevor wir mit der Analyse beginnen.

**Warum haben die beiden Datensätze unterschiedlich viele Zeilen?**
- `pvout_gemeinden.csv` hat 2132 Gemeinden (alle Gemeinden mit PVOUT-Daten aus Notebook 1)
- `df_h4_processed.csv` hat ca. 2100 Gemeinden (aus ElCom-Daten)

Nicht jede Gemeinde ist in beiden Datensätzen vorhanden. Mögliche Gründe:
- Gemeindefusionen seit dem Erhebungsjahr der ElCom-Daten (BFS-Nummern haben sich geändert)
- Kleine Gemeinden ohne eigenen Netzbetreiber in den ElCom-Daten
- Fehlende oder unvollständige Meldungen einzelner Netzbetreiber

**Was passiert mit den fehlenden Gemeinden?**

Für das nationale Ranking (Schritt 11) werden nur Gemeinden verwendet, die in **beiden** Datensätzen vorhanden sind.
Das nennt man einen *Inner Join*. Gemeinden ohne Tarif-Daten erscheinen grau in der Karte und nicht im Ranking.

Für die **Adress-Analyse** (Schritt 10) gibt es einen Fallback: wenn keine Tarif-Daten gefunden werden,
wird der Schweizer Durchschnittstarif verwendet – mit einem entsprechenden Hinweis.

**Was der Code macht:**
- Zeigt die Anzahl Gemeinden in beiden Datensätzen
- Listet Gemeinden aus PVOUT, für die kein Tarif gefunden wird
- Prüft den Wertebereich beider Datensätze auf Plausibilität

In [77]:
print(f'PVOUT-Gemeinden      : {len(pvout)}')
print(f'Tarif-Gemeinden      : {len(tarife)}')

ohne_tarif = pvout[~pvout['gemeinde_nr'].isin(tarife['gemeindeNummer'])]
print(f'Ohne Tarif-Daten     : {len(ohne_tarif)} Gemeinden (werden aus dem Ranking ausgeschlossen)')
if len(ohne_tarif) > 0:
    print(ohne_tarif[['gemeinde_name', 'kanton_name']].head(10).to_string(index=False))

tarif_min = tarife['total'].min()
tarif_max = tarife['total'].max()
pvout_min = pvout['pvout_kwh_kwp_year'].min()
pvout_max = pvout['pvout_kwh_kwp_year'].max()

print()
print(f'Tarif-Bereich        : {tarif_min:.1f} - {tarif_max:.1f} Rp./kWh')
print(f'PVOUT-Bereich        : {pvout_min:.0f} - {pvout_max:.0f} kWh/kWp/Jahr')
print(f'Score-Bereich (grob) : {pvout_min * tarif_min:.0f} - {pvout_max * tarif_max:.0f} Rp/kWp/Jahr')

PVOUT-Gemeinden      : 2132
Tarif-Gemeinden      : 2123
Ohne Tarif-Daten     : 29 Gemeinden (werden aus dem Ranking ausgeschlossen)
        gemeinde_name kanton_name
       Zürichsee (ZH)      Zürich
          Triesenberg         NaN
         Schellenberg         NaN
              Ruggell         NaN
Lac de Neuchâtel (BE)        Bern
               Mauren         NaN
              Triesen         NaN
           Greifensee      Zürich
       Bielersee (BE)        Bern
               Schaan         NaN

Tarif-Bereich        : 9.6 - 43.6 Rp./kWh
PVOUT-Bereich        : 885 - 1485 kWh/kWp/Jahr
Score-Bereich (grob) : 8531 - 64777 Rp/kWp/Jahr


## Schritt 5: Gemeindegrenzen laden

**Ziel:** Polygone aller Schweizer Gemeinden laden, damit Adressen einer Gemeinde zugeordnet werden können (Schritt 9) und die Choropleth-Karte gezeichnet werden kann (Schritt 12).

**Vorgehen:** Die swissBOUNDARIES3D-Datei (swisstopo) wird lokal kopiert und der Layer `tlm_hoheitsgebiet` geladen.

**Was der Code macht:**
- Das GPKG wird auf den lokalen `/content/`-Speicher kopiert, weil Google Drive SQLite-Dateien (wie GPKG) nicht direkt öffnen kann
- `engine='fiona'` gibt den korrekten Treiber zum Öffnen von GPKG-Dateien an
- `to_crs('EPSG:4326')` konvertiert das Koordinatensystem von Schweizer LV95 (EPSG:2056, in Metern) zu WGS84 (EPSG:4326, in Grad) – dasselbe System wie PVOUT.tif und die Geocoding-Koordinaten

In [78]:
import geopandas as gpd
import requests # Import requests for downloading files
import os

GPKG_LOCAL = '/content/swissboundaries.gpkg'
if not os.path.exists(GPKG_LOCAL):
    print('Lade GPKG von GitHub auf lokalen Speicher...')
    try:
        response = requests.get(GPKG_PATH)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        with open(GPKG_LOCAL, 'wb') as f:
            f.write(response.content)
        print('Heruntergeladen')
    except requests.exceptions.RequestException as e:
        print(f"Fehler beim Herunterladen des GPKG: {e}")
        raise # Re-raise the exception to indicate a critical error
else:
    print('GPKG bereits lokal vorhanden.')

gemeinden_geo   = gpd.read_file(GPKG_LOCAL, layer='tlm_hoheitsgebiet', engine='fiona')
gemeinden_wgs84 = gemeinden_geo.to_crs('EPSG:4326')

print(f'Gemeinden geladen    : {len(gemeinden_wgs84)}')
print(f'Koordinatensystem    : {gemeinden_wgs84.crs}')
gemeinden_wgs84[['bfs_nummer', 'name']].head(3)

GPKG bereits lokal vorhanden.
Gemeinden geladen    : 2136
Koordinatensystem    : EPSG:4326


,bfs_nummer,name
0,131,Adliswil
1,3714,Rheinwald
2,5722,Grens


## Schritt 6: Adresse eingeben

**Ziel:** Eine beliebige Schweizer Adresse angeben, für die das PV-Potenzial berechnet und mit allen Schweizer Gemeinden verglichen werden soll.

**Vorgehen:** Die Variable `ADRESSE` anpassen, dann Schritte 7–13 von oben nach unten ausführen.

**Tipps:**
- Immer Ort und «Schweiz» anfügen: `Hauptstrasse 1, Bern, Schweiz`
- Zum Vergleich eine sonnige Bergadresse ausprobieren (z.B. Täsch – Südexposition im Mattertal)
- Im flachen Mittelland haben Nachbaradressen oft ähnliche PVOUT-Werte – das ist korrekt (keine Topografie)

In [79]:
# <- HIER ADRESSE ANPASSEN (dann Schritte 6-12 ausführen)
ADRESSE = 'Bundesplatz 3, Bern, Schweiz'

# Testadressen zum Vergleich:
# ADRESSE = 'Lugano, Schweiz'         # sonnig, Tessin
# ADRESSE = 'Tasch, Schweiz'          # sehr sonnig, Südexposition Mattertal
# ADRESSE = 'Zürich, Schweiz'         # typisches Mittelland
# ADRESSE = 'Basel, Schweiz'          # nördlichste Grossstadt
# ADRESSE = 'Randa, Schweiz'          # Alpental, eher schattig

## Schritt 7: Geocoding – Adresse zu Koordinaten

**Ziel:** Die eingegebene Adresse in geografische Koordinaten (Breitengrad, Längengrad) umwandeln.

**Vorgehen:** Zwei Geocoding-Dienste werden nacheinander versucht.

**Was der Code macht:**
1. **swisstopo API (Hauptquelle):** `api3.geo.admin.ch` kennt alle offiziellen Schweizer Adressen aus dem Gebäude- und Wohnungsregister (GWR). Liefert sehr präzise Koordinaten direkt in WGS84 (lat/lon).
2. **Nominatim/OpenStreetMap (Fallback):** Wird verwendet, wenn die swisstopo API keine Ergebnisse liefert – z.B. bei unvollständigen oder nicht offiziellen Adressen.

**Hinweis:** Die swisstopo API kennt nur Schweizer Adressen. Für ausländische Orte greift der Code automatisch auf Nominatim zurück.

In [80]:
import requests
from geopy.geocoders import Nominatim

def geocodiere_adresse(adresse):
    try:
        url = 'https://api3.geo.admin.ch/rest/services/api/SearchServer'
        params = {'type': 'locations', 'searchText': adresse, 'limit': 1, 'sr': '4326'}
        r = requests.get(url, params=params, timeout=10)
        results = r.json().get('results', [])
        if results:
            attrs = results[0]['attrs']
            return attrs['lat'], attrs['lon'], attrs.get('label', adresse), 'swisstopo'
    except Exception:
        pass
    try:
        gc = Nominatim(user_agent='bina-pv-projekt')
        ort = gc.geocode(adresse)
        if ort:
            return ort.latitude, ort.longitude, ort.address, 'Nominatim'
    except Exception:
        pass
    return None, None, None, None

lat, lon, gefunden, quelle = geocodiere_adresse(ADRESSE)

if not lat:
    print(f'Adresse nicht gefunden: {ADRESSE}')
    print('Tipp: Ort und Schweiz anfügen, z.B. Hauptstrasse 1, Bern, Schweiz')
else:
    print(f'Geocoding ({quelle}): {gefunden}')
    print(f'Koordinaten          : {lat:.5f}N, {lon:.5f}E')

Geocoding (swisstopo): Bundesplatz 3 <b>3011 Bern</b>
Koordinaten          : 46.94677N, 7.44419E


## Schritt 8: PVOUT-Pixelwert an der Adresse ablesen

**Ziel:** Den PVOUT-Wert (kWh/kWp/Jahr) am genauen Standort der eingegebenen Adresse aus dem Raster auslesen.

**Was der Code macht:**
- `rasterio.open(PVOUT_PATH)` öffnet das GeoTIFF
- `rowcol(transform, lon, lat)` berechnet aus den geografischen Koordinaten die Zeile und Spalte im Pixelgitter
- `src.read(1)[r, c]` liest den Pixelwert an dieser Position direkt aus dem Array

**Hinweis zur Genauigkeit:**  
Die Datei hat eine Auflösung von ca. 1 km × 1 km pro Pixel. Das bedeutet: Zwei Adressen im gleichen Quartier können denselben Pixelwert haben – das ist ein Auflösungseffekt der Datenquelle, kein Fehler.
Im flachen Mittelland sind auch Nachbarpixel ähnlich (kaum Topografie). Grosse Abweichungen entstehen in Bergregionen: Nord- vs. Südhang können 30–40 % auseinanderliegen.

In [81]:
import rasterio
from rasterio.transform import rowcol

with rasterio.open(PVOUT_PATH) as src:
    r, c = rowcol(src.transform, lon, lat)
    pvout_adresse = float(src.read(1)[r, c])

print(f'PVOUT an der Adresse : {pvout_adresse:.0f} kWh/kWp/Jahr')
print('(Pixelaufloesung ca. 1 km x 1 km - Nachbaradressen haben denselben Wert)')

PVOUT an der Adresse : 1254 kWh/kWp/Jahr
(Pixelaufloesung ca. 1 km x 1 km - Nachbaradressen haben denselben Wert)


## Schritt 9: Gemeinde der Adresse bestimmen (Spatial Join)

**Ziel:** Herausfinden, zu welcher Gemeinde die eingegebene Adresse gehört.

**Was der Code macht:**
- `Point(lon, lat)` erstellt ein Punkt-Objekt aus den Koordinaten
- `gpd.sjoin(..., predicate='within')` prüft für jeden Punkt, in welchem Gemeindepolygon er liegt
- Das Ergebnis gibt BFS-Nummer und Gemeindename zurück
- Wenn der Punkt ausserhalb aller Polygone liegt (z.B. auf einem See), wird `None` gesetzt

**Warum brauchen wir die Gemeinde?**  
Mit der BFS-Nummer können wir in Schritt 10 den zugehörigen Stromtarif aus `df_h4_processed.csv` nachschlagen.

In [82]:
from shapely.geometry import Point

punkt   = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs='EPSG:4326')
treffer = gpd.sjoin(punkt, gemeinden_wgs84[['bfs_nummer', 'name', 'geometry']], how='left', predicate='within')

if len(treffer) > 0 and not treffer['name'].isna().all():
    gemeinde_nr   = int(treffer['bfs_nummer'].values[0])
    gemeinde_name = treffer['name'].values[0]
    print(f'Gemeinde             : {gemeinde_name} (BFS-Nr: {gemeinde_nr})')
else:
    gemeinde_nr   = None
    gemeinde_name = 'unbekannt'
    print('Gemeinde konnte nicht ermittelt werden (Koordinate ausserhalb CH-Grenzen?)')

Gemeinde             : Bern (BFS-Nr: 351)


## Schritt 10: Stromtarif nachschlagen und PV-Score berechnen

**Ziel:** Den Stromtarif der ermittelten Gemeinde nachschlagen und den PV-Attraktivitätsscore berechnen.

**Score-Formel:** `pv_score = PVOUT [kWh/kWp/Jahr] × Stromtarif [Rp./kWh]`  
**Einheit:** Rp/kWp/Jahr

**Beispielrechnung:**
- PVOUT = 1150 kWh/kWp/Jahr (typisches Mittelland)
- Tarif = 25 Rp./kWh
- Score = 28 750 Rp/kWp/Jahr
- 5-kWp-Anlage: 28 750 × 5 / 100 = **1437 CHF/Jahr Einsparung**

**Was passiert, wenn kein Tarif gefunden wird?**  
Wenn die Gemeinde nicht in `df_h4_processed.csv` vorhanden ist, wird der Schweizer Durchschnittstarif verwendet und ein Hinweis ausgegeben. Der Score ist dann eine Schätzung.

In [83]:
if gemeinde_nr is not None:
    tarif_zeile = tarife[tarife['gemeindeNummer'] == gemeinde_nr]
    if len(tarif_zeile) > 0:
        strompreis    = float(tarif_zeile['total'].values[0])
        netzbetreiber = tarif_zeile['netzbetreiber'].values[0]
    else:
        strompreis    = float(tarife['total'].mean())
        netzbetreiber = f'Schweizer Durchschnitt (kein Tarif fuer {gemeinde_name} gefunden)'
else:
    strompreis    = float(tarife['total'].mean())
    netzbetreiber = 'Schweizer Durchschnitt (Gemeinde nicht ermittelt)'

pv_score = pvout_adresse * strompreis

print(f'Netzbetreiber        : {netzbetreiber}')
print(f'Stromtarif H4        : {strompreis:.1f} Rp./kWh')
print(f'PVOUT Standort       : {pvout_adresse:.0f} kWh/kWp/Jahr')
print(f'PV-Score             : {pv_score:.0f} Rp/kWp/Jahr')
print()
print(f'  5-kWp-Anlage spart ca. {pv_score * 5 / 100:.0f} CHF/Jahr')

Netzbetreiber        : Energie Wasser Bern
Stromtarif H4        : 33.3 Rp./kWh
PVOUT Standort       : 1254 kWh/kWp/Jahr
PV-Score             : 41705 Rp/kWp/Jahr

  5-kWp-Anlage spart ca. 2085 CHF/Jahr


## Schritt 11: Nationaler Vergleich und Ranking (Qualitätskontrolle 2)

**Ziel:** Den PV-Score für alle Schweizer Gemeinden berechnen, ein Ranking erstellen und den Rang der eingegebenen Gemeinde im nationalen Vergleich zeigen.

**Was der Code macht:**
- Inner Join: Nur Gemeinden mit PVOUT- **und** Tarif-Daten werden berücksichtigt
- `pv_score = pvout_kwh_kwp_year × total` für jede Gemeinde
- Sortierung absteigend → beste Gemeinde = Rang 1

**Plausibilitätsprüfung der Rangliste:**

Die Top-Gemeinden sollten hohen PVOUT **und** hohen Tarif haben. Das sind typischerweise:
- Sonnige Gemeinden im Wallis oder Tessin, wenn dort die Tarife nicht sehr niedrig sind
- Oder Mittelland-Gemeinden mit teurem Strom, wenn ihre Sonneneinstrahlung ausreicht

Die Tabelle zeigt alle relevanten Spalten: `pvout_kwh_kwp_year × total = pv_score` – so kann man die Berechnung direkt nachvollziehen.

In [84]:
ranking = pvout.merge(
    tarife[['gemeindeNummer', 'total', 'netzbetreiber']],
    left_on='gemeinde_nr',
    right_on='gemeindeNummer',
    how='inner'
)
ranking['pv_score'] = ranking['pvout_kwh_kwp_year'] * ranking['total']
ranking = ranking.sort_values('pv_score', ascending=False).reset_index(drop=True)
ranking['rang'] = ranking.index + 1
total_gemeinden = len(ranking)

print(f'Gemeinden im Ranking : {total_gemeinden}')
print()

cols_show = ['rang', 'gemeinde_name', 'kanton_name', 'pvout_kwh_kwp_year', 'total', 'pv_score']
print('=== Top 10 – höchstes PV-Potenzial ===')
print(ranking[cols_show].head(10).to_string(index=False))

print()
print('=== Bottom 10 – niedrigstes PV-Potenzial ===')
print(ranking[cols_show].tail(10).to_string(index=False))

Gemeinden im Ranking : 2103

=== Top 10 – höchstes PV-Potenzial ===
 rang           gemeinde_name kanton_name  pvout_kwh_kwp_year  total     pv_score
    1              Kestenholz   Solothurn         1181.340234 43.613 51521.791642
    2        Teuffenthal (BE)        Bern         1223.059896 41.092 50257.977240
    3    Sils im Engadin/Segl  Graubünden         1382.785735 35.483 49065.386233
    4 Zihlschlacht-Sitterdorf     Thurgau         1181.185369 41.365 48859.732802
    5             Hohentannen     Thurgau         1181.132353 41.365 48857.539779
    6              Eggersriet  St. Gallen         1176.786849 41.463 48793.113118
    7      Hauptwil-Gottshaus     Thurgau         1175.078869 41.365 48607.137418
    8             Niederbüren  St. Gallen         1173.258621 41.365 48531.842845
    9            Bischofszell     Thurgau         1172.108801 41.365 48484.280535
   10    Belmont-sur-Lausanne       Waadt         1331.944824 36.037 47999.295630

=== Bottom 10 – niedrigstes P

In [85]:
mein_rang = ranking[ranking['gemeinde_nr'] == gemeinde_nr]['rang'].values

if len(mein_rang) > 0:
    rang = int(mein_rang[0])
    print(f'{gemeinde_name}: Rang {rang} von {total_gemeinden}')
    print(f'Besser als {100 * (1 - rang / total_gemeinden):.0f}% aller Schweizer Gemeinden')
else:
    rang = None
    print(f'{gemeinde_name}: nicht im Ranking (keine Tarif-Daten verfügbar)')
    print('Die Karte und Zusammenfassung zeigen den Score der Adresse ohne nationalen Rang.')

Bern: Rang 93 von 2103
Besser als 96% aller Schweizer Gemeinden


## Schritt 12: Interaktive Choropleth-Karte (Folium)

**Ziel:** Alle Schweizer Gemeinden auf einer interaktiven Karte nach ihrem PV-Score einfärben und die eingegebene Adresse markieren.

**Was der Code macht:**
- `karte_daten` verknüpft die Gemeindegeometrie mit den Score-Daten. Wichtig: Nur die benötigten Spalten werden übernommen – das verhindert einen Serialisierungsfehler, der entsteht, wenn Datum-Spalten aus dem GPKG in das Choropleth-Format exportiert werden.
- `folium.Choropleth(fill_color='RdYlGn')` färbt die Gemeinden: Rot = tiefer Score, Grün = hoher Score
- `nan_fill_color='lightgrey'` zeigt Gemeinden ohne Tarif-Daten grau
- `folium.Marker` setzt einen blauen Hausmarker an die eingegebene Adresse mit Score und Rang als Tooltip

**Interaktion:** Karte zoomen, verschieben, Marker anklicken für Details.

In [ ]:
import folium

# Nur benötigte Spalten übernehmen – vermeidet Timestamp-Serialisierungsfehler
karte_daten = gemeinden_wgs84[['bfs_nummer', 'name', 'geometry']].merge(
    ranking[['gemeinde_nr', 'pv_score', 'rang']],
    left_on='bfs_nummer',
    right_on='gemeinde_nr',
    how='left'
)
karte_daten = karte_daten[['bfs_nummer', 'name', 'pv_score', 'rang', 'geometry']].copy()

m = folium.Map(location=[46.8, 8.2], zoom_start=8, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=karte_daten.__geo_interface__,
    data=karte_daten,
    columns=['bfs_nummer', 'pv_score'],
    key_on='feature.properties.bfs_nummer',
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='PV-Score [Rp/kWp/Jahr]',
    nan_fill_color='lightgrey'
).add_to(m)

rang_text = f'Rang {rang} von {total_gemeinden}' if rang is not None else 'kein Rang (keine Tarif-Daten)'
folium.Marker(
    location=[lat, lon],
    tooltip=f'{ADRESSE} | Score: {pv_score:.0f} Rp/kWp/Jahr | {rang_text}',
    icon=folium.Icon(color='blue', icon='home')
).add_to(m)

m

## Schritt 13: Zusammenfassung

**Ziel:** Alle Ergebnisse für die eingegebene Adresse kompakt zusammenfassen.

**Was der Code macht:**
- Gibt PVOUT, Tarif, Score und Rang in einer übersichtlichen Tabelle aus
- Zeigt den nationalen Rang, falls die Gemeinde Tarif-Daten hat
- Berechnet die erwartete Jahreseinsparung für eine 5-kWp-Anlage

In [ ]:
print('=' * 50)
print(f'Adresse          : {ADRESSE}')
print(f'Gemeinde         : {gemeinde_name}')
print(f'PVOUT            : {pvout_adresse:.0f} kWh/kWp/Jahr')
print(f'Stromtarif H4    : {strompreis:.1f} Rp./kWh')
print(f'PV-Score         : {pv_score:.0f} Rp/kWp/Jahr')
if rang is not None:
    print(f'Nationaler Rang  : {rang} von {total_gemeinden}')
    print(f'5-kWp-Anlage     : ca. {pv_score * 5 / 100:.0f} CHF/Jahr Einsparung')
else:
    print('Nationaler Rang  : nicht verfügbar (keine Tarif-Daten)')
    print(f'5-kWp-Anlage     : ca. {pv_score * 5 / 100:.0f} CHF/Jahr Einsparung (Schaetzung)')
print('=' * 50)